# 01.8 保存、加载与推理（Saving, Loading, and Inference）

训练完模型之后，真正实用的问题是：  

- 怎么保存参数
- 怎么重新加载模型
- 怎么做推理

这一节就是把训练结果从“内存里的临时对象”变成“可以复用的模型”。  


## 学习目标

学完后你应该能

1. 理解 `state_dict` 是什么
2. 保存和加载模型参数
3. 保存和加载训练 checkpoint
4. 在推理前正确切到 `eval()`
5. 在推理中使用 `torch.no_grad()`
6. 让训练好的模型对新样本做预测

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
import torch.nn as nn

## 1. 准备一个小模型

这里先定义一个简单模型，并随便做几步训练，让参数不再是纯随机初值。  


In [ ]:
class SmallClassifier(nn.Module):
    def __init__(self, in_features=2, hidden_features=8, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(),
            nn.Linear(hidden_features, num_classes),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(1)
model = SmallClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()

X = torch.tensor(
    [
        [-1.0, -1.2],
        [-0.8, -0.5],
        [1.1, 0.9],
        [0.9, 1.2],
    ],
    dtype=torch.float32,
)
y = torch.tensor([0, 0, 1, 1], dtype=torch.long)

for _ in range(20):
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

print("training loss =", loss.item())

## 2. `state_dict` / `state_dict`

`state_dict` 是最常见的参数保存方式。  

dictionary，里面保存了层的参数张量。  


In [ ]:
state = model.state_dict()
print(type(state))
print(list(state.keys()))

## 3. 保存和加载模型参数

这里用临时目录演示，避免在仓库里留下测试文件。  


In [ ]:
with TemporaryDirectory() as tmp_dir:
    save_path = Path(tmp_dir) / "model_state.pt"
    torch.save(model.state_dict(), save_path)

    loaded_model = SmallClassifier()
    loaded_model.load_state_dict(torch.load(save_path))

    sample = torch.tensor([[0.8, 1.0]], dtype=torch.float32)
    out1 = model(sample)
    out2 = loaded_model(sample)

    print("save_path =", save_path)
    print("original output =", out1)
    print("loaded output =", out2)
    print("allclose =", torch.allclose(out1, out2))

重要点

- 保存的是参数（save parameters）
- 重新加载时，模型结构必须一致

## 4. 保存训练 checkpoint

只保存 `state_dict` 已经足够做推理。  

resume training，通常还会保存：  

- 优化器状态（optimizer state）
- 当前训练轮数（current epoch）
- 其他训练信息（other training metadata）

In [ ]:
with TemporaryDirectory() as tmp_dir:
    ckpt_path = Path(tmp_dir) / "checkpoint.pt"

    checkpoint = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": 20,
        "note": "demo checkpoint",
    }
    torch.save(checkpoint, ckpt_path)

    loaded_ckpt = torch.load(ckpt_path)

    resume_model = SmallClassifier()
    resume_optimizer = torch.optim.Adam(resume_model.parameters(), lr=0.05)
    resume_model.load_state_dict(loaded_ckpt["model_state"])
    resume_optimizer.load_state_dict(loaded_ckpt["optimizer_state"])

    print("loaded epoch =", loaded_ckpt["epoch"])
    print("loaded note =", loaded_ckpt["note"])

## 5. 推理

推理时的两个关键习惯

1. `model.eval()`
2. `with torch.no_grad():`

这两步都很重要。  


In [ ]:
model.eval()

new_x = torch.tensor(
    [
        [-1.1, -0.9],
        [1.2, 1.0],
        [0.2, 0.1],
    ],
    dtype=torch.float32,
)

with torch.no_grad():
    logits = model(new_x)
    probs = torch.softmax(logits, dim=1)
    preds = logits.argmax(dim=1)

print("logits =\n", logits)
print("probs =\n", probs)
print("preds =", preds)

这里的区分

- `logits`：原始分数（raw scores）
- `probs`：经过 `softmax` 后的概率
- `preds`：最终预测类别（final predicted class）

## 6. 一个简单预测函数

为了后续复用，通常会把推理逻辑封装成函数。  


In [ ]:
def predict_classes(model, x):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        preds = logits.argmax(dim=1)
    return preds


preds = predict_classes(model, new_x)
print(preds)

In [ ]:
# 练习 1
# 用一句话解释，为什么推理前通常要写 model.eval()。
# In one sentence, explain why we usually call model.eval() before inference.

参考回答

因为评估模式会让像 `Dropout`、`BatchNorm` 这样的层切换到正确的推理行为。  


In [ ]:
# 练习 2
# 补全下面的函数
#
# 目标
# 返回 softmax 概率

def predict_proba(model, x):
    # TODO
    pass


# print(predict_proba(model, new_x))

In [ ]:
# 练习 2 参考答案

def predict_proba_solution(model, x):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
    return probs


print(predict_proba_solution(model, new_x))

In [ ]:
# 练习 3
# 说明为什么只保存模型参数还不够支持断点续训。
# Explain why saving only model parameters is not enough to resume training.

参考回答

因为断点续训通常还需要优化器状态、当前 epoch 等信息，否则训练状态无法完整恢复。  


## 7. 小结

本节最重要的结论有三个：  

1. 保存模型最常见的是保存 `state_dict`
2. 推理前通常要 `eval()` + `no_grad()`
3. 断点续训通常要保存 checkpoint，而不仅是模型参数

你现在应该能回答

1. `state_dict` 为什么是常见保存格式？
2. 为什么加载时模型结构必须一致？
3. 为什么推理要避免梯度跟踪？

下一步建议

- 进入 Phase 1 小项目 notebook，把数据处理、模型、训练和推理完整串起来